# El juego de la vida con algoritmos geneticos y aplicaciones
## Mario Alejandro Castro Lerma

El Juego de la Vida **no se modifica**. Sus reglas son fijas e intocables — el algoritmo genetico opera por fuera.

La distinción clave es:

- El **Juego de la Vida** es el *entorno* (la función que evaluamos)
- El **Algoritmo Genético** evoluciona las *condiciones iniciales* de ese entorno

---

## ¿Qué evoluciona exactamente?

Cada individuo en la población es una **grilla inicial** — una foto del tablero en `t=0` antes de que el juego empiece. El AG nunca toca las reglas de Conway; solo decide qué células están vivas al principio.

```
Individuo = grilla binaria N×N

[ 0 1 0 0 1 ]
[ 1 0 0 1 0 ]   ← esto es el "cromosoma"
[ 0 0 1 0 1 ]
[ 1 1 0 0 0 ]
[ 0 1 0 1 1 ]
```

Desde ahí, el Juego de la Vida corre solo con sus reglas normales durante `t` pasos, y al final medimos el resultado con la función de fitness.

---

## El ciclo completo

```
Población inicial (grillas aleatorias)
        │
        ▼
┌─── Para cada individuo: ──────────────────────┐
│  1. Simular Conway durante t pasos            │
│  2. Medir fitness en el estado final          │
└───────────────────────────────────────────────┘
        │
        ▼
   Selección por torneo
   (los más aptos tienen más probabilidad de reproducirse)
        │
        ▼
   Cruzamiento uniforme
   (combinar dos grillas padre para generar un hijo)
        │
        ▼
   Mutación
   (invertir bits aleatorios del hijo)
        │
        ▼
   Nueva población
        │
        └──────────────────────────► repetir N generaciones
```

---

## Selección por torneo

Se eligen `k` individuos al azar de la población y gana el de mayor fitness. Ese ganador se convierte en padre. Se repite para elegir al segundo padre.

```python
# k=3: se comparan 3 individuos aleatorios, gana el mejor
idx = np.random.choice(len(population), k=3, replace=False)
winner = idx[np.argmax([fitnesses[i] for i in idx])]
```

Esto garantiza que los mejores individuos se reproduzcan más, pero sin eliminar completamente a los peores (diversidad genética).

---

## Cruzamiento uniforme

Dados dos padres, cada célula del hijo se hereda de uno u otro padre con probabilidad 50/50, como lanzar una moneda para cada bit:

```
Padre 1:  [ 1 0 1 1 0 0 1 0 ]
Padre 2:  [ 0 1 0 1 1 0 0 1 ]
Máscara:  [ 1 0 1 0 1 1 0 1 ]   ← bits aleatorios

Hijo:     [ 1 1 1 1 1 0 1 1 ]
            ↑   ↑       ↑       ← de padre 1
              ↑     ↑           ← de padre 2
```

El resultado es una grilla que mezcla regiones de ambos padres.

---

## Mutación

Después del cruzamiento, cada bit del hijo puede invertirse con probabilidad `p`. Esto introduce variación que el cruzamiento no puede generar por sí solo — es la única forma de que aparezca un `1` en una posición que era `0` en *ambos* padres.


---

## Elitismo

Los `n` mejores individuos de la generación actual pasan directamente a la siguiente sin modificarse. Esto evita que el AG pierda accidentalmente la mejor solución encontrada hasta el momento por efecto del azar.

---

## ¿Por qué funciona en el Juego de la Vida?

El Juego de la Vida tiene una propiedad que lo hace perfecto para esto: **sensibilidad a condiciones iniciales**. Cambiar una sola célula en `t=0` puede producir un resultado radicalmente diferente en `t=20`. Esto crea un paisaje de fitness complejo con miles de óptimos locales — exactamente el tipo de problema donde la búsqueda exhaustiva es imposible y los algoritmos genéticos brillan.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─── Conway's Game of Life ───────────────────────────────────────────────────

def gol_step(grid):
    """Advance the Game of Life by one generation using convolution."""
    neighbors = sum(
        np.roll(np.roll(grid, i, 0), j, 1)
        for i in (-1, 0, 1) for j in (-1, 0, 1)
        if (i != 0 or j != 0)
    )
    return ((neighbors == 3) | (grid == 1) & (neighbors == 2)).astype(np.uint8)


def simulate(grid, steps):
    """Run GoL for `steps` generations. Returns list of all states."""
    states = [grid.copy()]
    for _ in range(steps):
        grid = gol_step(grid)
        states.append(grid.copy())
    return states

### Funciones de fitness:

- Crecimiento neto (fitness = (celulas finales / n) - (celulas iniciales / n)) — diferencia entre células vivas al final y al inicio. Premia patrones que crecen, no solo que sobreviven. Puede producir resultados más explosivos.
- Actividad total (fitness = Σ(estados[t] ≠ estados[t+1])   para t = 0..T-1) — suma todos los cambios de estado (célula viva↔muerta) a lo largo de toda la simulación. Premia patrones que siguen cambiando mucho, como osciladores o gliders.

In [ ]:
# ─── Fitness functions ────────────────────────────────────────────────────────


def fitness_growth(grid, steps=20):
    states = simulate(grid, steps)
    n = float(grid.size)
    initial = states[0].sum() / n  # proporción inicial [0,1]
    final   = states[-1].sum() / n  # proporción final   [0,1]
    # Penaliza starts casi vacíos (< 10% densidad)
    if initial < 0.10:
        return -1.0
    return float(final - initial)   # rango real: [-1, 1]



def fitness_activity(grid, steps=20):
    # Simula todos los pasos y guarda cada estado intermedio
    states = simulate(grid, steps)
    
    # Para cada par de estados consecutivos (t, t+1),
    # cuenta cuántas celdas cambiaron de valor (0→1 o 1→0)
    # states[i] != states[i+1] produce una matriz booleana,
    # np.sum la cuenta, y el generador lo hace para todos los pasos
    # Resultado: total de "eventos" de cambio en toda la simulación
    # Patrones estáticos o extintos puntúan bajo; osciladores y gliders alto
    return float(sum(
        np.sum(states[i] != states[i + 1])
        for i in range(len(states) - 1)  # itera sobre t=0..T-1
    ))


FITNESS_FUNCTIONS = {
    'Crecimiento neto':        fitness_growth,
    'Actividad total':         fitness_activity,
}

In [4]:
# ─── Genetic Algorithm ────────────────────────────────────────────────────────

def init_population(pop_size, grid_n, density=0.3):
    """Random binary grids with given live-cell density."""
    return [
        (np.random.rand(grid_n, grid_n) < density).astype(np.uint8)
        for _ in range(pop_size)
    ]


def tournament_selection(population, fitnesses, k=3):
    """Select one individual via k-tournament."""
    idx = np.random.choice(len(population), k, replace=False)
    winner = idx[np.argmax([fitnesses[i] for i in idx])]
    return population[winner].copy()


def uniform_crossover(p1, p2):
    """Uniform crossover: each cell inherited randomly from one parent."""
    mask = np.random.rand(*p1.shape) < 0.5
    child = np.where(mask, p1, p2)
    return child.astype(np.uint8)


def mutate(individual, rate):
    """Flip each bit independently with probability `rate`."""
    flip = np.random.rand(*individual.shape) < rate
    return np.bitwise_xor(individual, flip.astype(np.uint8))


def genetic_algorithm(
    fitness_fn,
    grid_n        = 20,
    pop_size      = 60,
    generations   = 50,
    mutation_rate = 0.02,
    elitism       = 2,
    crossover_prob= 0.8,
    gol_steps     = 20,
    seed          = None,
    callback      = None,
):
    """
    Run a genetic algorithm to evolve Conway's Game of Life initial states.

    Parameters
    ----------
    fitness_fn     : callable(grid, steps) -> float
    grid_n         : side length of the square grid
    pop_size       : number of individuals
    generations    : number of GA iterations
    mutation_rate  : per-bit flip probability
    elitism        : top-N individuals carried unchanged to next generation
    crossover_prob : probability of producing a child via crossover vs. cloning
    gol_steps      : GoL simulation steps used for fitness evaluation
    seed           : random seed for reproducibility
    callback       : optional callable(gen, best_fit, avg_fit, best_ind)

    Returns
    -------
    history : dict with keys 'best', 'avg', 'best_individuals'
    """
    if seed is not None:
        np.random.seed(seed)

    population = init_population(pop_size, grid_n)

    history = {
        'best':             [],
        'avg':              [],
        'best_individuals': [],
    }

    for gen in range(generations):
        fitnesses = [fitness_fn(ind, gol_steps) for ind in population]

        best_idx  = int(np.argmax(fitnesses))
        best_fit  = fitnesses[best_idx]
        avg_fit   = float(np.mean(fitnesses))

        history['best'].append(best_fit)
        history['avg'].append(avg_fit)
        history['best_individuals'].append(population[best_idx].copy())

        if callback:
            callback(gen, best_fit, avg_fit, population[best_idx])

        # Elitism: preserve top individuals
        sorted_idx = np.argsort(fitnesses)[::-1]
        new_pop    = [population[i].copy() for i in sorted_idx[:elitism]]

        # Fill the rest of the new population
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population, fitnesses)
            if np.random.rand() < crossover_prob:
                p2    = tournament_selection(population, fitnesses)
                child = uniform_crossover(p1, p2)
            else:
                child = p1.copy()
            child = mutate(child, mutation_rate)
            new_pop.append(child)

        population = new_pop

    return history

In [5]:
# ─── Interactive Dashboard ────────────────────────────────────────────────────

# ── widgets ──
w_fitness   = widgets.Dropdown(
    options=list(FITNESS_FUNCTIONS.keys()),
    value='Crecimiento neto',
    description='Fitness:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='280px')
)
w_grid      = widgets.IntSlider(value=20, min=10, max=30, step=2,
                                description='Grilla N×N:',
                                style={'description_width': '80px'},
                                layout=widgets.Layout(width='320px'))
w_pop       = widgets.IntSlider(value=60, min=20, max=200, step=10,
                                description='Población:',
                                style={'description_width': '80px'},
                                layout=widgets.Layout(width='320px'))
w_gens      = widgets.IntSlider(value=50, min=10, max=150, step=10,
                                description='Generaciones:',
                                style={'description_width': '80px'},
                                layout=widgets.Layout(width='320px'))
w_mut       = widgets.FloatSlider(value=0.02, min=0.001, max=0.15, step=0.005,
                                  description='Mutación:',
                                  readout_format='.3f',
                                  style={'description_width': '80px'},
                                  layout=widgets.Layout(width='320px'))
w_gol       = widgets.IntSlider(value=20, min=5, max=60, step=5,
                                description='Pasos GoL:',
                                style={'description_width': '80px'},
                                layout=widgets.Layout(width='320px'))
w_seed      = widgets.IntText(value=42, description='Semilla:',
                              style={'description_width': '80px'},
                              layout=widgets.Layout(width='180px'))
w_run       = widgets.Button(description='▶  Ejecutar AG',
                             button_style='success',
                             layout=widgets.Layout(width='160px', height='36px'))
w_anim      = widgets.Button(description='Animar',
                             button_style='info',
                             layout=widgets.Layout(width='160px', height='36px'),
                             disabled=True)
w_heat = widgets.Button(description='Heatmap',
                        button_style='warning',
                        layout=widgets.Layout(width='160px', height='36px'),
                        disabled=True)
w_progress  = widgets.IntProgress(value=0, min=0, max=100,
                                   description='',
                                   bar_style='success',
                                   layout=widgets.Layout(width='100%'))
w_status    = widgets.Label(value='Configura los parámetros y presiona Ejecutar.')
w_out       = widgets.Output()

# ── state ──
_state = {'history': None, 'gol_steps': 20}

# ── plot helper ──
def plot_results(history, gol_steps, fitness_name):
    best_ind   = history['best_individuals'][-1]
    best_score = history['best'][-1]
    final_state = simulate(best_ind, gol_steps)[-1]

    fig = plt.figure(figsize=(14, 5))
    gs  = GridSpec(1, 3, figure=fig, wspace=0.35)

    # convergence curve
    ax1 = fig.add_subplot(gs[0])
    gens = range(len(history['best']))
    ax1.plot(gens, history['best'], color='#2563eb', lw=2,   label='Mejor')
    ax1.plot(gens, history['avg'],  color='#93c5fd', lw=1.5, label='Promedio', ls='--')
    ax1.fill_between(gens, history['avg'], history['best'], alpha=0.12, color='#2563eb')
    ax1.set_xlabel('Generación')
    ax1.set_ylabel('Fitness')
    ax1.set_title('Convergencia del AG')
    ax1.legend(fontsize=9)
    ax1.grid(True, lw=0.4, alpha=0.5)
    ax1.spines[['top','right']].set_visible(False)

    # initial state of best individual
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(best_ind, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
    ax2.set_title(f'Mejor individuo\n(estado inicial, {int(best_ind.sum())} células)')
    ax2.axis('off')

    # final state after simulation
    ax3 = fig.add_subplot(gs[2])
    ax3.imshow(final_state, cmap='Greens', interpolation='nearest', vmin=0, vmax=1)
    ax3.set_title(f'Estado final (t={gol_steps})\nFitness: {best_score:.1f}')
    ax3.axis('off')

    fig.suptitle(f'AG · GoL  —  Objetivo: {fitness_name}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


def animate_best(history, gol_steps):
    best_ind = history['best_individuals'][-1]
    states   = simulate(best_ind, gol_steps)

    fig, ax = plt.subplots(figsize=(5, 5))
    img = ax.imshow(states[0], cmap='binary', interpolation='nearest', vmin=0, vmax=1)
    title = ax.set_title('t = 0')
    ax.axis('off')

    def update(frame):
        img.set_data(states[frame])
        title.set_text(f't = {frame}  |  vivas: {int(states[frame].sum())}')
        return img, title

    ani = animation.FuncAnimation(
        fig, update, frames=len(states), interval=150, blit=True, repeat=True
    )
    # Store in _state so Python's GC doesn't destroy the animation object
    _state['_ani'] = ani
    from IPython.display import HTML
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

def plot_heatmap(history):
    individuals = history['best_individuals']
    n_gens = len(individuals)
    grid_n = individuals[0].shape[0]

    # Acumular cuántas veces cada celda cambió entre generaciones consecutivas
    change_map = np.zeros((grid_n, grid_n), dtype=float)
    for i in range(1, n_gens):
        change_map += np.abs(individuals[i].astype(float) - individuals[i-1].astype(float))

    # Frecuencia de activación promedio por generación
    activation_map = np.mean([ind.astype(float) for ind in individuals], axis=0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # — cambios acumulados —
    im0 = axes[0].imshow(change_map, cmap='hot', interpolation='nearest')
    axes[0].set_title('Cambios acumulados entre generaciones\n(más brillante = más inestable)')
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    # — frecuencia de activación —
    im1 = axes[1].imshow(activation_map, cmap='YlOrRd', interpolation='nearest', vmin=0, vmax=1)
    axes[1].set_title('Frecuencia de activación promedio\n(más oscuro = célula viva más seguido)')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    # — mejor individuo final como referencia —
    im2 = axes[2].imshow(individuals[-1], cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
    axes[2].set_title('Mejor individuo final\n(referencia)')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    fig.suptitle('Análisis de estabilidad celular a lo largo de la evolución', fontsize=12)
    plt.tight_layout()
    plt.show()

# ── callbacks ──
def on_run(_):
    w_run.disabled  = True
    w_anim.disabled = True
    w_progress.value = 0

    fitness_name = w_fitness.value
    fitness_fn   = FITNESS_FUNCTIONS[fitness_name]
    total_gens   = w_gens.value

    def cb(gen, best_fit, avg_fit, best_ind):
        w_progress.value = int((gen + 1) / total_gens * 100)
        w_status.value   = (
            f'Generación {gen+1}/{total_gens}  ·  '
            f'Mejor: {best_fit:.2f}  ·  Promedio: {avg_fit:.2f}'
        )

    with w_out:
        clear_output(wait=True)
        print('Ejecutando...')

    history = genetic_algorithm(
        fitness_fn    = fitness_fn,
        grid_n        = w_grid.value,
        pop_size      = w_pop.value,
        generations   = total_gens,
        mutation_rate = w_mut.value,
        gol_steps     = w_gol.value,
        seed          = w_seed.value,
        callback      = cb,
    )

    _state['history']   = history
    _state['gol_steps'] = w_gol.value

    with w_out:
        clear_output(wait=True)
        plot_results(history, w_gol.value, fitness_name)

    w_status.value   = f'✓ Listo. Mejor fitness final: {history["best"][-1]:.2f}'
    w_run.disabled   = False
    w_anim.disabled  = False
    w_heat.disabled = False


def on_animate(_):
    if _state['history'] is None:
        return
    with w_out:
        clear_output(wait=True)
        w_status.value = 'Generando animación...'
        animate_best(_state['history'], _state['gol_steps'])
        w_status.value = '✓ Animación lista. Usa los controles para reproducir.'
        
def on_heatmap(_):
    if _state['history'] is None:
        return
    with w_out:
        clear_output(wait=True)
        plot_heatmap(_state['history'])

w_heat.on_click(on_heatmap)


w_run.on_click(on_run)
w_anim.on_click(on_animate)


# ── layout ──
col1 = widgets.VBox([w_fitness, w_grid, w_pop, w_gens])
col2 = widgets.VBox([w_mut, w_gol, w_seed,
                     widgets.HBox([w_run, w_anim, w_heat])])

ui = widgets.VBox([
    widgets.HBox([col1, col2], layout=widgets.Layout(gap='32px')),
    w_progress,
    w_status,
    w_out,
])

display(ui)

---
**Áreas donde el enfoque es directamente aplicable:**

**Simulación de propagación** — incendios forestales, epidemias, contaminación en suelo. El autómata celular modela la propagación, y el AG optimiza estrategias de intervención: dónde colocar cortafuegos, cómo distribuir recursos médicos.

**Arquitectura y urbanismo computacional** — optimizar la distribución de espacios en un plano (qué celdas son muros, ventanas, zonas verdes) para maximizar flujo de personas o eficiencia energética.

**Criptografía** — los autómatas celulares tienen propiedades caóticas útiles para generar secuencias pseudoaleatorias.

---
